# ULGF Phase 0B baseline smoke test
This notebook mounts Drive, creates an isolated Python 3.7 environment, validates paths with the safe CLI, and generates a minimal baseline sample. Select a GPU runtime before starting. Update only the configuration cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
REPO_DIR = Path('/content/drive/MyDrive/ULGF-main')
CHECKPOINT = REPO_DIR / 'assets/checkpoint/final'
IMAGE_DIR = REPO_DIR / 'assets/smoke/images'
LABEL_DIR = REPO_DIR / 'assets/smoke/labels'
OUTPUT_DIR = REPO_DIR / 'outputs/phase0-baseline'
for name, path in {'repo': REPO_DIR, 'checkpoint': CHECKPOINT, 'images': IMAGE_DIR, 'labels': LABEL_DIR}.items():
    print(name, path, 'OK' if path.exists() else 'MISSING')


In [ ]:
!nvidia-smi
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh -O /content/miniconda37.sh
!bash /content/miniconda37.sh -b -p /content/miniconda37
!/content/miniconda37/bin/python --version


In [ ]:
!/content/miniconda37/bin/python -m pip install --no-cache-dir torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
!/content/miniconda37/bin/python -m pip install --no-cache-dir -r "{REPO_DIR / 'requirements-inference.txt'}"


In [ ]:
import subprocess
python = '/content/miniconda37/bin/python'
subprocess.run([python, '-c', "import sys, torch, diffusers, transformers; print(sys.version); print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available()); print('diffusers', diffusers.__version__); print('transformers', transformers.__version__)"], check=True)


In [ ]:
command = [
    '/content/miniconda37/bin/python',
    str(REPO_DIR / 'scripts/generate_image_baseline.py'),
    '--checkpoint', str(CHECKPOINT), '--image-dir', str(IMAGE_DIR),
    '--label-dir', str(LABEL_DIR), '--output-dir', str(OUTPUT_DIR),
    '--seed', '0', '--samples-per-layout', '1', '--dry-run'
]
subprocess.run(command, cwd=str(REPO_DIR), check=True)


If the dry run succeeds, remove the final `--dry-run` element below and execute the generation cell. Existing output is rejected by default; use a new output directory for repeated experiments.

In [ ]:
generation_command = command[:-1]
subprocess.run(generation_command, cwd=str(REPO_DIR), check=True)
print('Outputs:', sorted(str(path) for path in OUTPUT_DIR.rglob('*') if path.is_file()))
